<a href="https://colab.research.google.com/github/azka-asif/flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/azka-asif/flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

In [20]:
!pip -q install duckdb

In [21]:
import duckdb
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute("SET enable_progress_bar = false")

con.execute("""
    CREATE SECRET hf (
        TYPE huggingface,
        TOKEN ?
    )
""", [hf_token])

print("DuckDB connected to Hugging Face.")

DuckDB connected to Hugging Face.


In [22]:
REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"

print("Warehouse path set.")

Warehouse path set.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

For this task, the underlying performance data is recorded at the content page × client × reporting date level. I will use March 2026 as the development and verification period, and can aggregate the daily observations to the content page × client × month level when constructing the modelling dataset.


In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q1 = """
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items,
    COUNT(DISTINCT CAST(report_date AS DATE)) AS dates,
    COUNT(DISTINCT (
        client_hash_id,
        content_hash_id,
        CAST(report_date AS DATE)
    )) AS unique_grain_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.execute(q1).df()

,rows,clients,content_items,dates,unique_grain_rows
0,9841378,55,331437,31,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

features: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, scroll_events

label: field will be constructed from the observed performance data

context: report_date, month, client_hash_id, content_hash_id

excluded: client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available, gsc_sum_position. The boolean fields describe data-source availability rather than content performance and are therefore treated as metadata/context rather than model features. gsc_sum_position is excluded because it is a less interpretable aggregate than gsc_avg_position and is largely redundant for this task.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The March 2026 partition contains 9,841,378 observations and covers the full period from March 1 through March 31, 2026. This verifies the selected development window.


In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q2 = """
SELECT
    COUNT(*) AS row_count,
    MIN(CAST(report_date AS DATE)) AS first_date,
    MAX(CAST(report_date AS DATE)) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.execute(q2).df()


,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [26]:
q3 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content_pages,
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS unique_client_content_dates,
    COUNT(DISTINCT (client_hash_id, content_hash_id, month)) AS unique_client_content_months
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.execute(q3).df()

,total_rows,unique_content_pages,unique_clients,unique_client_content_dates,unique_client_content_months
0,9841378,331437,55,9841378,331437


The query confirms that each row represents a unique client × content page × reporting date observation: the total row count (9,841,378) matches the number of unique client-content-date combinations. There are 331,437 unique client-content-month combinations, meaning the March daily data can be aggregated to the intended monthly content-page level for modelling.


In [27]:
q4 = """
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (WHERE gsc_impressions IS NULL) AS null_gsc_impressions,
    COUNT(*) FILTER (WHERE gsc_clicks IS NULL) AS null_gsc_clicks,
    COUNT(*) FILTER (WHERE gsc_avg_position IS NULL) AS null_gsc_avg_position,

    COUNT(*) FILTER (WHERE ga4_pageviews IS NULL) AS null_ga4_pageviews,
    COUNT(*) FILTER (WHERE ga4_sessions IS NULL) AS null_ga4_sessions,
    COUNT(*) FILTER (WHERE ga4_users IS NULL) AS null_ga4_users,
    COUNT(*) FILTER (WHERE ga4_engaged_sessions IS NULL) AS null_ga4_engaged_sessions,

    COUNT(*) FILTER (WHERE sessions_organic IS NULL) AS null_sessions_organic,
    COUNT(*) FILTER (WHERE sessions_ai IS NULL) AS null_sessions_ai,

    COUNT(*) FILTER (WHERE scroll_events IS NULL) AS null_scroll_events
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.execute(q4).df()

,total_rows,null_gsc_impressions,null_gsc_clicks,null_gsc_avg_position,null_ga4_pageviews,null_ga4_sessions,null_ga4_users,null_ga4_engaged_sessions,null_sessions_organic,null_sessions_ai,null_scroll_events
0,9841378,0,0,6230317,3018741,3018741,3018741,3018741,3018741,3018741,3018741


The missing-value check shows that `gsc_impressions` and `gsc_clicks` are fully populated, while `gsc_avg_position` has some missingness. The GA4 and session-level fields also have missing values in approximately 31% of observations. These fields will  require appropriate missing-data handling during modelling rather than being excluded just because they contain null values. The availability flags will be used to differentiate genuine missing data from cases where a data source is unavailable.


In [28]:
q5 = """
SELECT
    client_has_gsc,
    client_has_ga4,
    gsc_data_available,
    ga4_data_available,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    client_has_gsc,
    client_has_ga4,
    gsc_data_available,
    ga4_data_available
ORDER BY row_count DESC
"""

con.execute(q5).df()

,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,row_count
0,True,True,False,False,4690323
1,True,True,True,False,1718348
2,True,False,True,<NA>,1528366
3,True,False,False,<NA>,1490375
4,True,True,True,True,364347
5,True,True,False,True,49619


The availability breakdown shows that missing values are strongly related to whether the corresponding data source is available. Many observations have GA4 or GSC data unavailable at the client or observation level. This means the missingness is not simply a random data-quality issue; it reflects differences in source coverage across observations. I will retain the relevant performance fields as candidate features and account for source availability when handling missing values.


In [29]:
q6 = """
SELECT
    COUNT(*) AS monthly_rows,
    COUNT(DISTINCT (client_hash_id, content_hash_id, month)) AS unique_client_content_months
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.execute(q6).df()

,monthly_rows,unique_client_content_months
0,9841378,331437


The March data contains 331,437 unique client × content × month combinations across 9,841,378 daily observations. This confirms that the daily performance table can be aggregated to the intended client × content page × month modelling grain, with each monthly unit summarizing the corresponding daily observations.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset provides observed content-performance signals, but it does not provide a complete history of what happened to each page or why it happened.

**Unbalanced history:** Not every client or content page necessarily has the same amount of historical data or the same level of measurement coverage. This limits direct comparisons between pages with different histories.

**Uneven data-source coverage:** GSC and GA4 availability varies across clients and observations. Some rows therefore contain search or analytics signals while others do not. The model cannot assume that missing values represent zero performance.

**GSC-only history:** Some observations have GSC data available without GA4 data. These rows provide search-performance information but cannot fully describe downstream traffic or engagement.

**Window overlap:** The March development window contains daily observations that will be aggregated into monthly content-page units. Performance measures within the same month are therefore derived from overlapping daily observations rather than independent experiments.

**No causal explanation:** The data records observed performance but does not tell us why a page performed poorly or whether refreshing the page would actually improve its performance. A model trained on this data can identify patterns associated with opportunity, but it cannot establish that a refresh will cause an improvement.

**No explicit refresh outcome:** There is no direct field indicating whether a page was refreshed or whether a refresh subsequently improved performance. Therefore, a refresh-opportunity label must be treated as a proxy rather than a directly observed ground truth.

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.